In [65]:
// hide-on-readme
import { readFileSync, writeFileSync } from 'fs';

type NotebookCell = {
  cell_type: string;
  source: string[] | string;
  id?: string;
  metadata?: Record<string, unknown>;
};

type NotebookFile = {
  cells: NotebookCell[];
  metadata?: Record<string, unknown>;
  nbformat?: number;
  nbformat_minor?: number;
};

function sourceLines(source: string[] | string): string[] {
  return Array.isArray(source) ? source : source.split('\n');
}

function markdownAnchor(text: string): string {
  return text
    .trim()
    .replace(/[\`*_{}[\]()#+.!?:;,\/\\|"']/g, '')
    .replace(/\s+/g, '-');
}

/**
 * Rebuild the notebook Table of Contents from markdown headings.
 *
 * Usage:
 * - updateNotebookToc();
 * - updateNotebookToc('notebook.ipynb');
 */
function updateNotebookToc(notebookPath = 'notebook.ipynb'): void {
  const raw = readFileSync(notebookPath, 'utf-8');
  const notebook = JSON.parse(raw) as NotebookFile;

  const tocCellIndex = notebook.cells.findIndex((cell) => {
    if (cell.cell_type !== 'markdown') {
      return false;
    }
    return sourceLines(cell.source).some((line) => line.trim() === '## Table of Contents');
  });

  if (tocCellIndex === -1) {
    throw new Error('Could not find a markdown cell titled "## Table of Contents".');
  }

  const headings: Array<{ level: number; text: string }> = [];
  for (let i = 0; i < notebook.cells.length; i += 1) {
    if (i === tocCellIndex) {
      continue;
    }

    const cell = notebook.cells[i];
    if (cell.cell_type !== 'markdown') {
      continue;
    }

    for (const line of sourceLines(cell.source)) {
      const match = line.match(/^(#{2,6})\s+(.+?)\s*$/);
      if (!match) {
        continue;
      }

      const level = match[1].length;
      const text = match[2].trim();
      if (text.toLowerCase() === 'table of contents') {
        continue;
      }

      headings.push({ level, text });
    }
  }

  if (headings.length === 0) {
    throw new Error('No markdown headings found to build the table of contents.');
  }

  const minLevel = Math.min(...headings.map((heading) => heading.level));
  const tocLines = [
    '## Table of Contents',
    '',
    ...headings.map((heading) => {
      const depth = Math.max(heading.level - minLevel, 0);
      const indent = '  '.repeat(depth);
      return `${indent}- [${heading.text}](#${markdownAnchor(heading.text)})`;
    }),
  ];

  notebook.cells[tocCellIndex].source = tocLines.map((line) => `${line}\n`);
  writeFileSync(notebookPath, `${JSON.stringify(notebook, null, 4)}\n`, { encoding: 'utf-8' });
}

(globalThis as Record<string, unknown>).updateNotebookToc = updateNotebookToc;

[Function: updateNotebookToc]


In [66]:
// hide-on-readme
import { execSync } from 'child_process';

/**
 * Update README.md from the notebook.
 *
 * Usage:
 * - updateReadme();
 * - updateReadme({ hideCode: true });
 */
function updateReadme(options: { hideCode?: boolean } = {}): void {
  const hideCode = options.hideCode ?? false;
  const command = hideCode ? './make.sh readme --hide-code' : './make.sh readme';

  console.log(`Running: ${command}`);
  void execSync(command, { stdio: 'pipe', encoding: 'utf-8' });
}

(globalThis as Record<string, unknown>).updateReadme = updateReadme;

[Function: updateReadme]


In [67]:
// hide-on-readme
import { existsSync, mkdirSync, writeFileSync, chmodSync } from 'fs';
import { join } from 'path';

/**
 * Install a git pre-commit hook that refreshes README.md before commit.
 *
 * Usage:
 * - addPreCommitHook();
 * - addPreCommitHook({ hideCode: false });
 */
function addPreCommitHook(options: { hideCode?: boolean } = {}): void {
  const hideCode = options.hideCode ?? true;
  const readmeCommand = hideCode ? './make.sh readme --hide-code' : './make.sh readme';
  const hookDir = join('.git', 'hooks');
  const hookPath = join(hookDir, 'pre-commit');

  if (!existsSync('.git')) {
    throw new Error('No .git directory found. Run this from the repository root.');
  }

  if (!existsSync(hookDir)) {
    mkdirSync(hookDir, { recursive: true });
  }

  const hookScript = [
    '#!/usr/bin/env bash',
    'set -euo pipefail',
    '',
    readmeCommand,
    '',
    'git add README.md',
  ].join('\n');

  writeFileSync(hookPath, `${hookScript}\n`, { encoding: 'utf-8' });
  chmodSync(hookPath, 0o755);
}

// Auto Exec
addPreCommitHook();
updateNotebookToc();
updateReadme({ hideCode: true });

Running: ./make.sh readme --hide-code


# GRAPHNB - The Graph Based Notebook

This is a personal project to collect my personal knowledge and small utilities into an obsidian inspired jupyter notebook.

To view the interactive version of this project, please visit [freyground.com](https://freyground.com).

## Table of Contents

- [License](#License)
- [Packages](#Packages)
  - [Common ECS Interface](#Common-ECS-Interface)
  - [Pico ECS](#Pico-ECS)
- [Articles](#Articles)
- [Notes](#Notes)


## License

This project is licensed under the GNU Affero General Public License v3.0.
See [LICENSE](LICENSE) for the full text.

## Packages

### Common ECS Interface

The common ECS Interface implements a shared interface between multiple ECS implementations

In [68]:
// hide-on-readme
type CommonEntityId = number;
type CommonSystem<TWorld extends CommonECS = CommonECS> = (ecs: TWorld, dt: number) => void;

interface CommonECS {
  createEntity(): CommonEntityId;
  destroyEntity(entity: CommonEntityId): void;
  addComponent<T>(entity: CommonEntityId, name: string, value: T): void;
  getComponent<T>(entity: CommonEntityId, name: string): T | undefined;
  hasComponents(entity: CommonEntityId, names: string[]): boolean;
  query(names: string[]): CommonEntityId[];
  addSystem(system: CommonSystem): void;
  tick(dt: number): void;
}

### Pico ECS

Pico ECS is an extremely minimal ECS implementation for interactive applications.

In [69]:
// hide-on-readme
// A minimal Entity-Component-System (ECS) implementation in TypeScript.

// TODO: Usage examples

type EntityId = number;
type ComponentMap<T = unknown> = Map<EntityId, T>;
type System = (ecs: ECS, dt: number) => void;

class ECS {
  private nextEntityId = 1;
  private alive = new Set<EntityId>();
  private components = new Map<string, ComponentMap>();
  private systems: System[] = [];

  createEntity(): EntityId {
    const id = this.nextEntityId++;
    this.alive.add(id);
    return id;
  }

  destroyEntity(entity: EntityId): void {
    this.alive.delete(entity);
    for (const store of this.components.values()) {
      store.delete(entity);
    }
  }

  addComponent<T>(entity: EntityId, name: string, value: T): void {
    if (!this.alive.has(entity)) {
      throw new Error(`Entity ${entity} does not exist`);
    }
    let store = this.components.get(name);
    if (!store) {
      store = new Map<EntityId, T>();
      this.components.set(name, store as ComponentMap);
    }
    (store as ComponentMap<T>).set(entity, value);
  }

  getComponent<T>(entity: EntityId, name: string): T | undefined {
    return (this.components.get(name) as ComponentMap<T> | undefined)?.get(entity);
  }

  hasComponents(entity: EntityId, names: string[]): boolean {
    return names.every((name) => this.components.get(name)?.has(entity));
  }

  query(names: string[]): EntityId[] {
    const out: EntityId[] = [];
    for (const entity of this.alive) {
      if (this.hasComponents(entity, names)) {
        out.push(entity);
      }
    }
    return out;
  }

  addSystem(system: System): void {
    this.systems.push(system);
  }

  tick(dt: number): void {
    for (const system of this.systems) {
      system(this, dt);
    }
  }
}

// publish-skip
// Expose ECS as a global for easy access in other cells
type PicoECSPublic = {
  ECS: typeof ECS;
};
type PicoEntity = EntityId;
type PicoSystem = System;

const PicoECS: PicoECSPublic = { ECS };
(globalThis as Record<string, unknown>).PECS = PicoECS;


{ ECS: [class ECS] }


## Articles

## Notes